<a href="https://colab.research.google.com/github/ridasaeed297/Dot_Catcher/blob/main/Maze_Solver_using_A_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import heapq
import math
import random
import time
from enum import Enum


# ============================================================
# CELL TYPES
# ============================================================

class Cell(Enum):
    EMPTY = 0
    WALL = 1
    START = 2
    GOAL = 3


# ============================================================
# A* MAZE SOLVER
# ============================================================

class AStarMazeSolver:

    def __init__(
        self,
        maze,
        start,
        goal,
        allow_diagonal=False,
        heuristic_type="manhattan"
    ):
        self.maze = maze
        self.rows = len(maze)
        self.cols = len(maze[0])

        self.start = start
        self.goal = goal

        self.allow_diagonal = allow_diagonal
        self.heuristic_type = heuristic_type

        # g(n): cost from start
        self.g_score = {}

        # f(n): g(n) + h(n)
        self.f_score = {}

        # Parent of each node
        self.came_from = {}

        # Nodes that have been completely explored
        self.closed_set = set()

        # Nodes discovered but not completely explored
        self.open_set = []

        # Statistics
        self.nodes_expanded = 0
        self.nodes_generated = 0

    # --------------------------------------------------------
    # HEURISTICS
    # --------------------------------------------------------

    def heuristic(self, current, goal):

        r1, c1 = current
        r2, c2 = goal

        dx = abs(r1 - r2)
        dy = abs(c1 - c2)

        if self.heuristic_type == "manhattan":
            return dx + dy

        elif self.heuristic_type == "euclidean":
            return math.sqrt(dx ** 2 + dy ** 2)

        elif self.heuristic_type == "chebyshev":
            return max(dx, dy)

        else:
            raise ValueError("Unknown heuristic")

    # --------------------------------------------------------
    # VALID CELL
    # --------------------------------------------------------

    def is_valid(self, row, col):

        return (
            0 <= row < self.rows
            and
            0 <= col < self.cols
            and
            self.maze[row][col] != Cell.WALL.value
        )

    # --------------------------------------------------------
    # GET NEIGHBORS
    # --------------------------------------------------------

    def get_neighbors(self, current):

        row, col = current

        # Normal movement
        directions = [
            (-1, 0, 1.0),    # Up
            (1, 0, 1.0),     # Down
            (0, -1, 1.0),    # Left
            (0, 1, 1.0),     # Right
        ]

        # Diagonal movement
        if self.allow_diagonal:
            diagonal_cost = math.sqrt(2)

            directions += [
                (-1, -1, diagonal_cost),
                (-1, 1, diagonal_cost),
                (1, -1, diagonal_cost),
                (1, 1, diagonal_cost),
            ]

        neighbors = []

        for dr, dc, cost in directions:

            nr = row + dr
            nc = col + dc

            if self.is_valid(nr, nc):

                # Prevent diagonal movement through corners
                if dr != 0 and dc != 0:

                    if (
                        not self.is_valid(row + dr, col)
                        or
                        not self.is_valid(row, col + dc)
                    ):
                        continue

                neighbors.append(
                    ((nr, nc), cost)
                )

        return neighbors

    # --------------------------------------------------------
    # RECONSTRUCT PATH
    # --------------------------------------------------------

    def reconstruct_path(self):

        path = []

        current = self.goal

        while current in self.came_from:

            path.append(current)

            current = self.came_from[current]

        path.append(self.start)

        path.reverse()

        return path

    # --------------------------------------------------------
    # A* ALGORITHM
    # --------------------------------------------------------

    def solve(self):

        # Initial values
        self.g_score[self.start] = 0

        self.f_score[self.start] = (
            self.heuristic(
                self.start,
                self.goal
            )
        )

        # Heap entry:
        # (f_score, counter, node)

        counter = 0

        heapq.heappush(
            self.open_set,
            (
                self.f_score[self.start],
                counter,
                self.start
            )
        )

        self.nodes_generated += 1

        while self.open_set:

            current_f, _, current = heapq.heappop(
                self.open_set
            )

            # Ignore stale heap entries
            if current in self.closed_set:
                continue

            # Goal reached
            if current == self.goal:

                path = self.reconstruct_path()

                return {
                    "found": True,
                    "path": path,
                    "cost": self.g_score[current],
                    "nodes_expanded": self.nodes_expanded,
                    "nodes_generated": self.nodes_generated
                }

            self.closed_set.add(current)

            self.nodes_expanded += 1

            # Explore neighbors
            for neighbor, movement_cost in self.get_neighbors(
                current
            ):

                if neighbor in self.closed_set:
                    continue

                tentative_g = (
                    self.g_score[current]
                    +
                    movement_cost
                )

                # Better path found
                if (
                    neighbor not in self.g_score
                    or
                    tentative_g < self.g_score[neighbor]
                ):

                    self.came_from[neighbor] = current

                    self.g_score[neighbor] = tentative_g

                    h = self.heuristic(
                        neighbor,
                        self.goal
                    )

                    self.f_score[neighbor] = (
                        tentative_g + h
                    )

                    counter += 1

                    heapq.heappush(
                        self.open_set,
                        (
                            self.f_score[neighbor],
                            counter,
                            neighbor
                        )
                    )

                    self.nodes_generated += 1

        # No path exists
        return {
            "found": False,
            "path": None,
            "cost": math.inf,
            "nodes_expanded": self.nodes_expanded,
            "nodes_generated": self.nodes_generated
        }


# ============================================================
# CONSOLE VISUALIZER
# ============================================================

def display_maze(
    maze,
    start,
    goal,
    path=None,
    explored=None
):

    path = set(path or [])
    explored = set(explored or [])

    print()

    for r in range(len(maze)):

        line = ""

        for c in range(len(maze[0])):

            position = (r, c)

            if position == start:
                symbol = "S"

            elif position == goal:
                symbol = "G"

            elif maze[r][c] == Cell.WALL.value:
                symbol = "#"

            elif position in path:
                symbol = "*"

            elif position in explored:
                symbol = "."

            else:
                symbol = " "

            line += symbol + " "

        print(line)

    print()


# ============================================================
# RANDOM MAZE GENERATOR
# ============================================================

def generate_maze(
    rows,
    cols,
    wall_probability=0.25
):

    maze = [
        [
            Cell.WALL.value
            if random.random() < wall_probability
            else Cell.EMPTY.value
            for _ in range(cols)
        ]
        for _ in range(rows)
    ]

    return maze


# ============================================================
# EXAMPLE
# ============================================================

if __name__ == "__main__":

    random.seed(42)

    rows = 15
    cols = 30

    maze = generate_maze(
        rows,
        cols,
        wall_probability=0.25
    )

    start = (0, 0)
    goal = (rows - 1, cols - 1)

    # Make sure start and goal are accessible
    maze[start[0]][start[1]] = Cell.START.value
    maze[goal[0]][goal[1]] = Cell.GOAL.value

    print("=" * 60)
    print("A* MAZE SOLVER")
    print("=" * 60)

    print("\nOriginal Maze:")

    display_maze(
        maze,
        start,
        goal
    )

    # Create solver
    solver = AStarMazeSolver(
        maze=maze,
        start=start,
        goal=goal,
        allow_diagonal=False,
        heuristic_type="manhattan"
    )

    # Run A*
    result = solver.solve()

    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    if result["found"]:

        print("PATH FOUND!")

        print(
            f"Path length: "
            f"{len(result['path']) - 1}"
        )

        print(
            f"Path cost: "
            f"{result['cost']:.2f}"
        )

        print(
            f"Nodes expanded: "
            f"{result['nodes_expanded']}"
        )

        print(
            f"Nodes generated: "
            f"{result['nodes_generated']}"
        )

        print("\nShortest Path:")

        display_maze(
            maze,
            start,
            goal,
            path=result["path"],
            explored=solver.closed_set
        )

        print("Coordinates:")
        print(result["path"])

    else:

        print("NO PATH EXISTS!")

        print(
            f"Nodes expanded: "
            f"{result['nodes_expanded']}"
        )

        display_maze(
            maze,
            start,
            goal,
            explored=solver.closed_set
        )

A* MAZE SOLVER

Original Maze:

S #   #       #   # #   # #     #     #       #     # #     
                      # #   # # #         #         #   #   
            # #     #                 #           #     # # 
#       #           #           #             #           # 
        #   #         #     #   #     #           #     #   
        # #     # # #   #     # #   # #           # #       
      #       # #       #         #           #   #   #     
                #           #       #   # #     # #   #     
        #   #           #       # #           #           # 
        # # #         #           #     #             # #   
        # #           # # #         #           #           
      #     #         # # #                   #   #     #   
                #             #                     #       
        #             #       # #             # # # #     # 
    #     #   #     #   #     # #                         G 

PATH FOUND!
Path length: 45
Path cost: 45.00
Nodes e